# 27 QH Three-Model Comparison (Observed-QH New Metrics)

## 1. Methodological context
All three candidates are mixed-frequency quarter-hour forecasting strategies evaluated on observed quarter-hour targets only.

- `qh-fs1__lear__hourly_anchor__lear_fs3_combo_promoted`
- `qh-fs1__xgboost__hourly_anchor__xgboost_fs3_combo_pruned_candidate`
- `qh-fs1__mean_shape__hourly_anchor__lear_strict`

`qh-fs1__mean_shape__hourly_anchor__lear_strict` uses LEAR_STRICT hourly anchors (phase07 NL bridge aligned export) plus a zero-mean quarter-hour deviation so each hour-average forecast equals the hourly anchor.

Primary three-model ranking scope is `STRICT_THREE_MODEL_QH_OVERLAP`.
Observed-QH scoring uses observed targets only (`y_true` non-null).


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidate_roots if (p / '.git').exists() or (p / 'AGENTS.md').exists()), Path.cwd())
OUT_ROOT = PROJECT_ROOT / 'data/02_Forecasting/01_DA_prices/quarterhour_da/finalisation_runs/qh_three_model_comparison'
assert OUT_ROOT.exists(), f'Missing output root: {OUT_ROOT}'

run_dirs = sorted([p for p in OUT_ROOT.iterdir() if p.is_dir() and (p / 'comparison_summary.json').exists()])
assert run_dirs, f'No comparison run dirs found under {OUT_ROOT}'
RUN_DIR = run_dirs[-1]
print('Using run:', RUN_DIR)
summary = json.loads((RUN_DIR / 'comparison_summary.json').read_text(encoding='utf-8'))
summary

## 2. Artifact loading

In [ ]:
comparison_table = pd.read_csv(RUN_DIR / 'comparison_table_by_scope.csv')
conventional = pd.read_csv(RUN_DIR / 'conventional_metrics_by_scope.csv')
operational = pd.read_csv(RUN_DIR / 'operational_metrics_by_scope.csv')
support_scope = pd.read_csv(RUN_DIR / 'support_by_scope.csv')
support_split = pd.read_csv(RUN_DIR / 'support_by_split.csv')
support_lead = pd.read_csv(RUN_DIR / 'support_by_lead_day.csv')

print('comparison rows:', comparison_table.shape[0])
print('conventional rows:', conventional.shape[0])
print('operational rows:', operational.shape[0])

## 3. Support diagnostics

In [ ]:
display(support_scope)
display(support_split.sort_values(['comparison_scope', 'dataset_split']))
display(support_lead.sort_values(['comparison_scope', 'dataset_split', 'lead_day']))

## 4. Conventional metrics comparison

In [ ]:
conv = conventional.copy()
strict_conv = conv[conv['comparison_scope'] == 'STRICT_THREE_MODEL_QH_OVERLAP'].copy()
display(strict_conv.sort_values(['dataset_split', 'lead_day', 'mae', 'model_id']))

for split in ['validation', 'test']:
    part = strict_conv[strict_conv['dataset_split'] == split].copy()
    if part.empty:
        continue
    piv = part.pivot(index='lead_day', columns='model_id', values='mae').sort_index()
    ax = piv.plot(kind='bar', figsize=(11,4), title=f'MAE by lead_day ({split}) - STRICT_THREE_MODEL_QH_OVERLAP')
    ax.set_ylabel('MAE')
    plt.tight_layout()
    plt.show()

metric_cols = ['mae','rmse','bias','median_ae','p90_ae','p95_ae','rmae']
display(strict_conv[['comparison_scope','dataset_split','lead_day','model_id'] + [c for c in metric_cols if c in strict_conv.columns]].sort_values(['dataset_split','lead_day','mae']))

## 5. Operational metrics comparison

In [ ]:
ops = operational.copy()
strict_ops = ops[ops['comparison_scope'] == 'STRICT_THREE_MODEL_QH_OVERLAP'].copy()
display(strict_ops.sort_values(['dataset_split', 'lead_day', 'model_id']))

for split in ['validation', 'test']:
    part = strict_ops[strict_ops['dataset_split'] == split].copy()
    if part.empty:
        continue
    for metric in ['top4_hit','top8_hit','top16_hit','bottom4_hit','bottom8_hit','bottom16_hit','spearman_daily_mean','regret_low_L8']:
        if metric not in part.columns:
            continue
        piv = part.pivot(index='lead_day', columns='model_id', values=metric).sort_index()
        ax = piv.plot(kind='bar', figsize=(11,4), title=f'{metric} by lead_day ({split}) - STRICT_THREE_MODEL_QH_OVERLAP')
        ax.set_ylabel(metric)
        plt.tight_layout()
        plt.show()

## 6. Selected visual diagnostics

In [ ]:
from datetime import date

phase27_run = Path(summary['phase27_run_dir'])
model3_run = Path(summary['model3_run_dir'])
phase27_pred = pd.read_csv(phase27_run / 'predictions_long.csv', low_memory=False)
model3_pred = pd.read_csv(model3_run / 'predictions_long.csv', low_memory=False)

keep_two = [
    'qh-fs1__lear__hourly_anchor__lear_fs3_combo_promoted',
    'qh-fs1__xgboost__hourly_anchor__xgboost_fs3_combo_pruned_candidate',
]
two = phase27_pred[phase27_pred['model'].isin(keep_two)].copy()
two['model_id'] = two['model'].astype(str)
two['y_true'] = pd.to_numeric(two['y_true'], errors='coerce')
two['y_pred'] = pd.to_numeric(two['y_pred'], errors='coerce')
two['forecast_origin_utc'] = pd.to_datetime(two['forecast_origin_utc'], utc=True, errors='coerce')
two['target_timestamp_utc'] = pd.to_datetime(two['target_timestamp_utc'], utc=True, errors='coerce')
two['lead_day'] = pd.to_numeric(two['lead_day'], errors='coerce').astype('Int64')
two['dataset_split'] = two['dataset_split'].astype(str)

m3 = model3_pred.copy()
m3['model_id'] = m3.get('model_id', m3.get('model'))
m3 = m3[m3['model_id'] == 'qh-fs1__mean_shape__hourly_anchor__lear_strict'].copy()
m3['y_true'] = pd.to_numeric(m3['y_true'], errors='coerce')
m3['y_pred'] = pd.to_numeric(m3['y_pred'], errors='coerce')
m3['forecast_origin_utc'] = pd.to_datetime(m3['forecast_origin_utc'], utc=True, errors='coerce')
m3['target_timestamp_utc'] = pd.to_datetime(m3['target_timestamp_utc'], utc=True, errors='coerce')
m3['lead_day'] = pd.to_numeric(m3['lead_day'], errors='coerce').astype('Int64')
m3['dataset_split'] = m3['dataset_split'].astype(str)

pred = pd.concat([
    two[['model_id','dataset_split','forecast_origin_utc','target_timestamp_utc','lead_day','y_true','y_pred']],
    m3[['model_id','dataset_split','forecast_origin_utc','target_timestamp_utc','lead_day','y_true','y_pred']],
], ignore_index=True)

key_cols = ['dataset_split','forecast_origin_utc','target_timestamp_utc','lead_day']
valid = pred[pred['y_true'].notna() & pred['y_pred'].notna()].copy()
counts = valid.groupby(key_cols)['model_id'].nunique().reset_index(name='n_models')
keys = counts[counts['n_models'] == 3][key_cols].copy()
strict = valid.merge(keys, on=key_cols, how='inner')
strict['target_local'] = strict['target_timestamp_utc'].dt.tz_convert('Europe/Amsterdam')
strict['target_local_date'] = strict['target_local'].dt.date

daily = strict.groupby(['dataset_split','target_local_date'], dropna=False).agg(
    y_true_min=('y_true','min'),
    y_true_max=('y_true','max'),
    y_true_mean=('y_true','mean'),
    spread=('y_true', lambda s: float(s.max() - s.min())),
).reset_index()

selected = {}
val_days = daily[daily['dataset_split'] == 'validation'].sort_values('target_local_date')
test_days = daily[daily['dataset_split'] == 'test'].sort_values('target_local_date')
if not val_days.empty:
    selected['validation_day'] = val_days.iloc[0]['target_local_date']
if not test_days.empty:
    selected['test_day'] = test_days.iloc[0]['target_local_date']
if not daily.empty:
    selected['volatile_day'] = daily.sort_values('spread', ascending=False).iloc[0]['target_local_date']
neg_days = daily[daily['y_true_min'] < 0].sort_values('y_true_min')
if not neg_days.empty:
    selected['low_negative_day'] = neg_days.iloc[0]['target_local_date']

selected

In [ ]:
def plot_day(day_value):
    day_frame = strict[strict['target_local_date'] == day_value].copy()
    if day_frame.empty:
        return
    fig, ax = plt.subplots(figsize=(12,4))
    true_path = day_frame[['target_local','y_true']].drop_duplicates().sort_values('target_local')
    ax.plot(true_path['target_local'], true_path['y_true'], label='realised', linewidth=2, color='black')
    for model_id, g in day_frame.groupby('model_id'):
        gg = g.sort_values('target_local')
        ax.plot(gg['target_local'], gg['y_pred'], label=model_id, alpha=0.9)
    ax.set_title(f'STRICT_THREE_MODEL_QH_OVERLAP day: {day_value}')
    ax.set_ylabel('EUR/MWh')
    ax.legend(loc='best', fontsize=8)
    plt.tight_layout()
    plt.show()

for name, d in selected.items():
    print(name, d)
    plot_day(d)

## 7. Model ranking and interpretation

In [ ]:
strict_conv = conventional[(conventional['comparison_scope']=='STRICT_THREE_MODEL_QH_OVERLAP') & conventional['dataset_split'].isin(['validation','test'])].copy()
strict_ops = operational[(operational['comparison_scope']=='STRICT_THREE_MODEL_QH_OVERLAP') & operational['dataset_split'].isin(['validation','test'])].copy()
support_w = support_lead[support_lead['comparison_scope']=='STRICT_THREE_MODEL_QH_OVERLAP'][['dataset_split','lead_day','n_quarters']].copy()
support_w = support_w.rename(columns={'n_quarters':'support_n_quarters'})

def weighted(df, value_col):
    m = df.copy()
    if 'support_n_quarters' not in m.columns:
        m = m.merge(support_w, on=['dataset_split','lead_day'], how='left')
    weight_col = 'n_quarters' if 'n_quarters' in m.columns else 'support_n_quarters'
    out = (
        m.groupby(['dataset_split','model_id'])
         .apply(lambda g: float((g[value_col] * g[weight_col]).sum() / g[weight_col].sum()), include_groups=False)
         .reset_index(name=f'weighted_{value_col}')
    )
    return out

w_mae = weighted(strict_conv, 'mae')
display(w_mae.sort_values(['dataset_split','weighted_mae']))

if not strict_ops.empty:
    for col in ['top8_hit','spearman_daily_mean','regret_low_L8']:
        if col in strict_ops.columns:
            w = weighted(strict_ops, col)
            print(f'Weighted {col}')
            display(w.sort_values(['dataset_split', f'weighted_{col}'], ascending=[True, col not in ['top8_hit','spearman_daily_mean']]))

## 8. Thesis-ready wording

On observed quarter-hour targets, three mixed-frequency strategies were compared on strict shared support (`STRICT_THREE_MODEL_QH_OVERLAP`).
The LEAR_STRICT-based quarter-hour strategy (`qh-fs1__mean_shape__hourly_anchor__lear_strict`) uses LEAR_STRICT hourly level forecasts and a zero-mean intra-hour deviation component, preserving the hourly anchor by construction.
LEAR_STRICT hourly selection remains overlap-based from the hourly three-way evidence, so claims are restricted to shared-support performance rather than full-period dominance.
Final quarter-hour ranking should therefore be reported on strict common observed-QH support with observed-target-only scoring.
